# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import joblib

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [ ]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour.astype(int)
        df['dayofweek'] = df['timestamp'].dt.weekday
        df = df.drop(columns=['timestamp'])
        return df

In [ ]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column='dayofweek'):
        self.target_column = target_column
        self.encoder = OneHotEncoder(sparse=False, handle_unknown='ignore', dtype=int)
        self.cat_cols = None

    def fit(self, X, y=None):
        cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.target_column in cols:
            cols.remove(self.target_column)
        self.cat_cols = cols
        if len(self.cat_cols) > 0:
            self.encoder.fit(X[self.cat_cols])
        return self

    def transform(self, X):
        df = X.copy()
        if len(self.cat_cols) > 0:
            encoded = self.encoder.transform(df[self.cat_cols])
            names = self.encoder.get_feature_names(input_features=self.cat_cols)
            encoded_df = pd.DataFrame(encoded, columns=names, index=df.index)
            df = df.drop(columns=self.cat_cols)
            df = pd.concat([df, encoded_df], axis=1)

        if self.target_column in df.columns:
            y = df.pop(self.target_column)
            return df, y
        return df, None

In [ ]:
class TrainValidationTest:
    def split(self, X, y):
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.2, random_state=21, stratify=y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_tv, y_tv, test_size=0.2, random_state=21, stratify=y_tv
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [ ]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results_ = []
        self.best_estimators_ = {}

    def choose(self, X_train, y_train, X_valid, y_valid):
        best_valid_acc = -1
        best_name = None

        for idx, gs in enumerate(tqdm(self.grids, desc="Models", leave=True)):
            name = self.grid_dict[idx]
            print(f"Estimator: {name}")

            gs.fit(X_train, y_train)

            best_params = gs.best_params_
            best_cv_score = round(gs.best_score_, 3)
            y_pred_v = gs.predict(X_valid)
            valid_acc = accuracy_score(y_valid, y_pred_v)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_cv_score}")
            print(f"Validation set accuracy score for best params: {valid_acc:.3f} \n")

            self.results_.append([name, best_params, round(valid_acc, 6)])
            self.best_estimators_[name] = gs.best_estimator_

            if valid_acc > best_valid_acc:
                best_valid_acc = valid_acc
                best_name = name

        print(f"Classifier with best validation set accuracy: {best_name}")
        return best_name

    def best_results(self):
        df = pd.DataFrame(self.results_, columns=['model', 'params', 'valid_score'])
        return df

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [ ]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        acc = accuracy_score(y_test, self.estimator.predict(X_test))
        print(f"Accuracy of the final model is {acc}")
        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print("The model was successfully saved")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [ ]:
csv_path = "../data/checker_submits.csv"
df = pd.read_csv(csv_path)

In [ ]:
preprocessing = Pipeline([
    ('extractor', FeatureExtractor()),
    ('ohe', MyOneHotEncoder(target_column='dayofweek'))
])

In [ ]:
X_ohe, y = preprocessing.fit_transform(df)

In [ ]:
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X_ohe, y)

In [ ]:
svm_params = [{'kernel': ('linear', 'rbf', 'sigmoid'),
               'C': [0.01, 0.1, 1, 1.5, 5, 10],
               'gamma': ['scale', 'auto'],
               'class_weight': ('balanced', None),
               'random_state': [21],
               'probability': [True]}]

tree_params = [{'criterion': ('gini', 'entropy'),
                'max_depth': [10, 15, 20, 21, 22, 23, 24, 25, 30, None],
                'class_weight': ('balanced', None),
                'random_state': [21]}]

rf_params = [{'criterion': ('gini', 'entropy'),
              'max_depth': [15, 20, 22, 24, 30, None],
              'n_estimators': [50, 100],
              'class_weight': ('balanced', None),
              'random_state': [21]}]

In [ ]:
gs_svm = GridSearchCV(SVC(), param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=-1)

In [ ]:
grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}
selector = ModelSelection(grids, grid_dict)

In [ ]:
best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)

In [ ]:
selector.best_results()

In [ ]:
if best_model_name == 'SVM':
    best_estimator = gs_svm.best_estimator_
elif best_model_name == 'Decision Tree':
    best_estimator = gs_tree.best_estimator_
else:
    best_estimator = gs_rf.best_estimator_

In [ ]:
final = Finalize(best_estimator)

In [ ]:
test_acc = final.final_score(X_train, y_train, X_test, y_test)

In [ ]:
model_short = best_model_name.lower().replace(' ', '_')
filename = f"{model_short}_{test_acc}.sav"
final.save_model(filename)

In [ ]:
loaded = joblib.load(filename)
final_loaded = Finalize(loaded)
loaded_acc = final_loaded.final_score(X_train, y_train, X_test, y_test)
print(f"Scores match after loading: {test_acc == loaded_acc}")

In [ ]:
test_df = pd.DataFrame({
    'uid': ['u1', 'u2'],
    'labname': ['labA', 'labB'],
    'numTrials': [1, 2],
    'timestamp': ['2020-04-17 10:00:00', '2020-04-18 14:30:00']
})
pipe_test = Pipeline([('feat', FeatureExtractor()), ('ohe', MyOneHotEncoder(target_column='labname'))])
X_t, y_t = pipe_test.fit_transform(test_df)
print("Target column was categorical and was NOT one-hot encoded (kept as y):", y_t.name)